# RMB Fly Behavior Analysis — Python Pipeline (Notebook View)

This notebook is a thin wrapper around the modular `rmb` Python package. Each cell:

1. Runs **one step** of the pipeline by calling the corresponding `rmb.steps.stepNN_*.run(...)` function.
2. Prints the **input** and **output** schema (column names, dtypes, shape, summary stats).
3. Renders the **review plots** (PNG inline + a link to the interactive HTML).

The same code is also runnable as a single command:

```bash
python -m rmb.pipeline --config configs/default.yaml
```

Both produce identical output under `test/output/<run_id>/<NN_step>/`.

In [ ]:
import sys, os, json
from pathlib import Path
from IPython.display import Image, Markdown, display

REPO = Path.cwd()
sys.path.insert(0, str(REPO / 'src'))

from rmb.config import load_config
from rmb.steps import (
    step01_load, step02_metadata, step03_process, step04_sleep,
    step05_heatmaps, step06_frequency, step07_stats, step08_summary,
)

config = load_config(str(REPO / 'configs' / 'default.yaml'))
print('run_id   :', config.run_id)
print('run_dir  :', config.run_dir)

def show_schema(step_dir):
    with open(os.path.join(step_dir, 'schema.json')) as f:
        s = json.load(f)
    md = [f"**Schema** \u2014 produced by `{s['produced_by']}` at `{s['produced_at']}`\n"]
    for a in s['artifacts']:
        md.append(f"- `{os.path.basename(a['path'])}` ({a.get('format','?')})")
        if 'shape' in a:
            md.append(f"  - shape: `{a['shape']}`, summary: `{a.get('summary', {})}`")
    display(Markdown('\n'.join(md)))

def show_plots(plot_paths):
    for p in plot_paths:
        if p.endswith('.png'):
            display(Image(filename=p, embed=True))
        elif p.endswith('.html'):
            rel = os.path.relpath(p, REPO)
            display(Markdown(f'[Interactive plot \u2192 `{rel}`]({rel})'))

## Step 1 — Load Monitor*.txt

**Input**: `config.data_dir/Monitor*.txt` (TAB-separated, no header). Column 7 marks `MT`/`CT`/`Pn`; columns 10+ are per-channel counts.

**Output**: `raw_monitors.parquet` (long-form: `monitor`, `record_type`, `timepoint`, `ch01..chN`) + `per_monitor_counts.csv`.

In [ ]:
r1 = step01_load.run(config)
show_schema(r1['step_dir'])
show_plots(r1['plots'])

## Step 2 — Metadata

**Input**: `config.metadata_dir/*_metadata.csv` (one row per fly: `Monitor_number`, `Channel`, `Group`, `Treatment`, `Sex`, ...).

**Output**: `metadata.parquet` (concatenated tidy table). If no metadata files exist, a placeholder is generated.

In [ ]:
r2 = step02_metadata.run(config, monitor_files=r1['monitor_files'])
show_schema(r2['step_dir'])
show_plots(r2['plots'])

## Step 3 — Process to wide MT/CT/PN

**Input**: `raw_monitors.parquet` from Step 1.

**Output**: `mt.parquet`, `ct.parquet`, `pn.parquet` — wide format, rows = timepoint, columns = `<Monitor>_Ch<N>` (one column per fly).

In [ ]:
r3 = step03_process.run(config, raw_path=r1['raw_path'])
show_schema(r3['step_dir'])
show_plots(r3['plots'])

## Step 4 — Sleep / awake

**Input**: `mt.parquet`, `pn.parquet` (Step 3), `config.sleep_threshold`.

**Output**: `awake.parquet` (1=awake), `sleep.parquet` (1=asleep), `pn_awake.parquet` (position masked by awake).

In [ ]:
r4 = step04_sleep.run(config, mt_path=r3['mt_path'], pn_path=r3['pn_path'])
show_schema(r4['step_dir'])
show_plots(r4['plots'])

## Step 5 — Heatmaps (review plots only)

**Input**: `mt`, `pn`, `sleep`, `pn_awake`.

**Output**: four PNG+HTML heatmap pairs. No new data artifacts.

In [ ]:
r5 = step05_heatmaps.run(
    config,
    mt_path=r3['mt_path'], pn_path=r3['pn_path'],
    sleep_path=r4['sleep_path'], pn_awake_path=r4['pn_awake_path'],
)
show_schema(r5['step_dir'])
show_plots(r5['plots'])

## Step 6 — Frequency distributions

**Input**: `pn_awake.parquet` + `metadata.parquet`.

**Output**: `freq_all.csv` (counts per position × channel), `freq_per_day.csv` (long-form), `freq_by_group.csv` (mean proportion per position per group).

In [ ]:
r6 = step06_frequency.run(
    config,
    pn_awake_path=r4['pn_awake_path'],
    metadata_path=r2['metadata_path'],
)
show_schema(r6['step_dir'])
show_plots(r6['plots'])

## Step 7 — Statistics

**Input**: `mt`, `awake`, `sleep`, `metadata`.

**Output**: `channel_stats.csv` (per-channel summary), `circadian_hourly.csv` (hourly population means), `stats_results.csv` (pairwise Mann–Whitney U across `Group`).

In [ ]:
r7 = step07_stats.run(
    config,
    mt_path=r3['mt_path'],
    awake_path=r4['awake_path'],
    sleep_path=r4['sleep_path'],
    metadata_path=r2['metadata_path'],
)
show_schema(r7['step_dir'])
show_plots(r7['plots'])

## Step 8 — Summary + run index

**Output**: `analysis_summary.json` (config + per-step artifact map) and `index.html` (one-page review linking every plot in this run).

In [ ]:
step_results = {
    '01_load': r1, '02_metadata': r2, '03_process': r3, '04_sleep': r4,
    '05_heatmaps': r5, '06_frequency': r6, '07_stats': r7,
}
r8 = step08_summary.run(config, step_results=step_results)
show_schema(r8['step_dir'])
rel_idx = os.path.relpath(r8['index_html'], REPO)
display(Markdown(f'Open the full review index \u2192 [`{rel_idx}`]({rel_idx})'))